# Advanced Inference Strategies

Beyond basic KV caching and continuous batching, modern LLM serving systems employ
sophisticated techniques to push throughput and latency further. This notebook covers
the most impactful strategies deployed in production today.

1. Speculative decoding — verify many tokens in one pass
2. Disaggregated inference — separate prefill and decode hardware
3. Prefix caching (RadixAttention) — reuse shared prompt computation
4. Cascade inference — route easy queries to small models
5. Multi-token prediction — predict multiple tokens per step
6. Early exit / adaptive computation
7. Comparison table

## 1. Speculative Decoding

### The problem
Decode is memory-bound at batch=1: we load the full weight matrix from HBM to compute
a single token. The GPU's compute units are mostly idle.

### The insight
Verifying K tokens is almost as cheap as generating 1 token — the target model can
score all K positions in parallel (like prefill). So: **draft cheaply, verify in bulk**.

### How it works

```
┌──────────────────────────────────────────────────────────────────┐
│  1. DRAFT: Small model generates K candidate tokens              │
│     (fast, low-quality, autoregressive)                          │
│                                                                  │
│     Input: "The capital of France is"                            │
│     Draft: ["Paris", ",", "which", "is", "known"]  (K=5)         │
│                                                                  │
│  2. VERIFY: Large model scores all K tokens in ONE forward pass  │
│     (parallel evaluation, like prefill)                          │
│                                                                  │
│     Accept/reject each token via modified rejection sampling:    │
│     - If P_target(token) ≥ P_draft(token) → ACCEPT              │
│     - Otherwise → accept with probability P_target/P_draft       │
│     - First rejection → resample from corrected distribution     │
│                                                                  │
│     Result: ["Paris" ✓, "," ✓, "which" ✓, "is" ✗] → 3 accepted  │
│                                                                  │
│  3. Output: 3 tokens generated in ~1 target model forward pass   │
│     (vs 3 separate forward passes without speculation)           │
└──────────────────────────────────────────────────────────────────┘
```

### Key property: mathematically lossless
The rejection sampling scheme guarantees the output distribution is **identical** to
standard sampling from the target model. Zero quality loss — just faster.

### Variants

| Variant | Draft mechanism | Speedup | Notes |
|---------|----------------|---------|-------|
| Standard (Leviathan 2023) | Separate small model | 2–2.5x | Requires maintaining two models |
| Medusa (Cai 2024) | Extra prediction heads on base model | 2.2–3.6x | No separate model; tree-based verification |
| EAGLE (Li 2024) | Feature-level autoregression | 2.7–3.5x | Lightweight; uses second-to-top layer features |
| Lookahead (Fu 2023) | Jacobi iteration + n-gram pools | 1.5–2x | No training; trades FLOPs for latency |

### When it helps vs hurts

**Helps**: Low-batch decode (memory-bound), predictable outputs (code, structured data),
well-aligned draft/target models.

**Hurts**: High-batch scenarios (already compute-saturated), domain mismatch (low acceptance
rate), very short outputs (overhead > savings).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate speculative decoding speedup based on acceptance rate
def speculative_speedup(acceptance_rate, draft_tokens_k, draft_cost_ratio=0.1):
    """
    Calculate speedup from speculative decoding.
    
    acceptance_rate: probability each drafted token is accepted
    draft_tokens_k: number of tokens drafted per round
    draft_cost_ratio: cost of draft model forward pass relative to target
    """
    # Expected accepted tokens per round (geometric distribution)
    expected_accepted = 0
    for i in range(draft_tokens_k):
        expected_accepted += acceptance_rate ** (i + 1)
    # +1 for the resampled token after first rejection
    tokens_per_round = expected_accepted + 1
    
    # Cost per round: K draft steps + 1 target verification
    cost_per_round = draft_tokens_k * draft_cost_ratio + 1.0
    
    # Without speculation: 1 token per target forward pass
    baseline_cost_per_token = 1.0
    
    spec_cost_per_token = cost_per_round / tokens_per_round
    return baseline_cost_per_token / spec_cost_per_token

# Plot speedup vs acceptance rate for different K values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

acceptance_rates = np.linspace(0.3, 0.99, 50)

for k in [3, 5, 7, 10]:
    speedups = [speculative_speedup(a, k) for a in acceptance_rates]
    ax1.plot(acceptance_rates, speedups, linewidth=2, label=f"K={k} draft tokens")

ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel("Acceptance rate")
ax1.set_ylabel("Speedup vs standard decoding")
ax1.set_title("Speculative decoding speedup")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.5, 6)

# Show tokens generated per verification round
for k in [3, 5, 7, 10]:
    tokens = [sum(a**(i+1) for i in range(k)) + 1 for a in acceptance_rates]
    ax2.plot(acceptance_rates, tokens, linewidth=2, label=f"K={k}")

ax2.set_xlabel("Acceptance rate")
ax2.set_ylabel("Expected tokens per round")
ax2.set_title("Tokens generated per verification step")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At 80% acceptance, K=5: {speculative_speedup(0.8, 5):.1f}x speedup")
print(f"At 90% acceptance, K=5: {speculative_speedup(0.9, 5):.1f}x speedup")
print(f"At 70% acceptance, K=5: {speculative_speedup(0.7, 5):.1f}x speedup")
print(f"\nHigher acceptance rates come from better draft-target alignment.")
print(f"Code generation often has >90% acceptance (predictable patterns).")

## 2. Disaggregated Inference — separate prefill and decode

### The mismatch problem

Prefill and decode have fundamentally different resource needs:

| | Prefill | Decode |
|---|---|---|
| Bottleneck | Compute | Memory bandwidth |
| GPU utilisation | High (large matmuls) | Low (tiny matmuls) |
| Ideal hardware | High FLOPS (H100 SXM) | High bandwidth (many cheaper GPUs) |
| Batch size needed | Small (already saturated) | Large (need batching to amortise) |

Co-locating them on the same GPU means:
- A long prefill stalls all decode requests (latency spikes)
- Can't independently scale or optimise each phase
- Resource allocation is a compromise for both

### The solution: physically separate them

```
┌─────────────────┐         KV cache transfer         ┌──────────────────┐
│  PREFILL POOL   │ ─────────────────────────────────► │  DECODE POOL     │
│                 │        (via NVLink/RDMA)           │                  │
│  • High-FLOP   │                                    │  • High-bandwidth│
│    GPUs         │                                    │    GPUs          │
│  • Small batch  │                                    │  • Large batch   │
│  • Compute-     │                                    │  • Memory-       │
│    saturated    │                                    │    saturated     │
└─────────────────┘                                    └──────────────────┘
```

### Systems

| System | Key innovation | Result |
|--------|---------------|--------|
| **Splitwise** (Microsoft) | Phase splitting + fast interconnect transfer | 1.4x throughput, 20% lower cost |
| **DistServe** (OSDI 2024) | Independent parallelism per phase | 7.4x more requests served |
| **Mooncake** (Moonshot/Kimi) | KV cache tiers (HBM → DRAM → SSD) | 525% throughput on long-context |

### Tradeoff
KV cache must be transferred between machines after prefill completes.
For a 2048-dim, 32-layer model at 4K context: ~512 MB per request.
Needs high-bandwidth interconnect (NVLink, InfiniBand) to not become the new bottleneck.

In [ ]:
# Simulate disaggregated vs co-located inference

def simulate_colocated(requests, prefill_rate=500, decode_rate=50):
    """Tokens/sec rates. Prefill blocks decode."""
    total_time = 0
    results = []
    
    for r in requests:
        # Prefill blocks everything
        prefill_time = r["prompt_len"] / prefill_rate
        decode_time = r["output_len"] / decode_rate
        total_time += prefill_time + decode_time
        results.append({"ttft": prefill_time * 1000, "total": (prefill_time + decode_time) * 1000})
    
    return results, total_time

def simulate_disaggregated(requests, prefill_rate=500, decode_rate=50, transfer_overhead=0.01):
    """Prefill and decode run on separate pools. Prefill doesn't block decode."""
    results = []
    # Prefill pool processes prompts; decode pool generates tokens
    # Key: decode pool is never stalled by prefill work
    decode_queue_time = 0
    
    for r in requests:
        prefill_time = r["prompt_len"] / prefill_rate + transfer_overhead
        decode_time = r["output_len"] / decode_rate
        results.append({"ttft": prefill_time * 1000, "total": (prefill_time + decode_time) * 1000})
    
    # Total time: prefill and decode can overlap
    total_prefill = sum(r["prompt_len"] / prefill_rate for r in requests)
    total_decode = sum(r["output_len"] / decode_rate for r in requests)
    # With disaggregation, these overlap
    total_time = max(total_prefill, total_decode) * 1.1  # small overhead
    
    return results, total_time

# Mixed workload: some short, some long prompts
mixed_requests = [
    {"prompt_len": 50, "output_len": 100},
    {"prompt_len": 4000, "output_len": 200},  # long RAG context
    {"prompt_len": 30, "output_len": 50},
    {"prompt_len": 100, "output_len": 300},
    {"prompt_len": 8000, "output_len": 150},  # very long context
    {"prompt_len": 60, "output_len": 80},
]

coloc_results, coloc_time = simulate_colocated(mixed_requests)
disagg_results, disagg_time = simulate_disaggregated(mixed_requests)

print(f"{'Metric':<25} {'Co-located':<15} {'Disaggregated':<15}")
print("-" * 55)
print(f"{'Total time (s)':<25} {coloc_time:<15.2f} {disagg_time:<15.2f}")
print(f"{'Avg TTFT (ms)':<25} {np.mean([r['ttft'] for r in coloc_results]):<15.1f} {np.mean([r['ttft'] for r in disagg_results]):<15.1f}")
print(f"{'Max TTFT (ms)':<25} {max(r['ttft'] for r in coloc_results):<15.1f} {max(r['ttft'] for r in disagg_results):<15.1f}")
print(f"\nDisaggregation is most impactful with mixed workloads (short + long prompts).")
print(f"The long prefills no longer block the short decode requests.")

## 3. Prefix Caching (RadixAttention)

### The observation
Many requests share common prefixes:
- System prompts (identical across all requests)
- Few-shot examples (same across a session)
- Multi-turn conversation history (grows incrementally)
- RAG document context (same doc retrieved for similar queries)

Without prefix caching, we redundantly compute identical KV cache entries for
every request that shares the same prefix.

### How RadixAttention works (SGLang)

```
                    Radix Tree of KV Caches
                    ========================
                    
                        [system prompt]
                       /               \
              [few-shot ex 1]      [few-shot ex 2]
              /           \              |
        [user Q1]    [user Q2]     [user Q3]
        
    Request A: system + few-shot 1 + Q1 → reuses 90% of KV cache
    Request B: system + few-shot 1 + Q2 → reuses 80% of KV cache  
    Request C: system + few-shot 2 + Q3 → reuses 70% of KV cache
```

The radix tree indexes cached KV entries by token sequence. Longest prefix match
determines how much computation can be skipped.

### Speedup
- **Up to 6.4x throughput** for workloads with heavy prefix sharing
- System prompts (often 500–2000 tokens) are computed once, reused for all requests
- Multi-turn chat: only the new user message needs prefill

### Systems
- SGLang (RadixAttention — original implementation)
- vLLM (automatic prefix caching)
- TensorRT-LLM (KV cache reuse)

In [ ]:
# Demonstrate prefix sharing savings

system_prompt_tokens = 800
few_shot_tokens = 400
user_query_tokens = 50

total_prompt = system_prompt_tokens + few_shot_tokens + user_query_tokens
num_requests = 100

# Without prefix caching: every request computes the full prompt
tokens_computed_no_cache = total_prompt * num_requests

# With prefix caching: system prompt computed once, few-shot once, only query per-request
tokens_computed_with_cache = (
    system_prompt_tokens +          # computed once
    few_shot_tokens +               # computed once  
    user_query_tokens * num_requests  # unique per request
)

savings = 1 - tokens_computed_with_cache / tokens_computed_no_cache

fig, ax = plt.subplots(figsize=(10, 4))

categories = ['System prompt\n(800 tokens)', 'Few-shot\n(400 tokens)', 'User queries\n(50 × 100)']
without_cache = [system_prompt_tokens * num_requests, few_shot_tokens * num_requests, user_query_tokens * num_requests]
with_cache = [system_prompt_tokens, few_shot_tokens, user_query_tokens * num_requests]

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, without_cache, width, label='Without prefix cache', color='red', alpha=0.7)
bars2 = ax.bar(x + width/2, with_cache, width, label='With prefix cache', color='green', alpha=0.7)

ax.set_ylabel('Tokens to compute')
ax.set_title(f'Prefill computation: {num_requests} requests sharing system prompt + few-shot')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Without prefix caching: {tokens_computed_no_cache:,} tokens computed")
print(f"With prefix caching:    {tokens_computed_with_cache:,} tokens computed")
print(f"Savings: {savings*100:.1f}% less prefill computation")
print(f"\nFor a typical chat API with 800-token system prompts,")
print(f"prefix caching eliminates ~{system_prompt_tokens/total_prompt*100:.0f}% of prefill work on every request.")

## 4. Cascade Inference — route to the right model

### The observation
Not every query needs a 405B parameter model. Simple questions ("What's 2+2?",
"Translate 'hello' to French") can be answered perfectly by a 7B model at 50x lower cost.

### How it works

```
                    ┌─────────────┐
     Query ──────►  │   ROUTER    │
                    │  (tiny, ~1ms)│
                    └──────┬──────┘
                           │
                    ┌──────┴──────┐
                    │             │
              ┌─────▼─────┐ ┌────▼──────┐
              │  Small     │ │  Large     │
              │  Model     │ │  Model     │
              │  (7B)      │ │  (70B+)    │
              │  ~80% of   │ │  ~20% of   │
              │  queries   │ │  queries   │
              └────────────┘ └────────────┘
```

### RouteLLM (2024)
- Router trained on human preference data to predict query difficulty
- Routes easy queries to a weak model, hard queries to a strong model
- **2x+ cost reduction** with no quality degradation on benchmarks
- Router generalises when underlying models are swapped

### Tradeoff
- Requires training/maintaining a router
- Risk of routing a hard query to the small model (quality regression)
- Adds routing latency (though typically <5ms)
- More complex infrastructure (multiple model deployments)

In [ ]:
# Simulate cascade routing cost savings

def simulate_cascade(num_queries=1000, easy_ratio=0.75, 
                     small_cost=1.0, large_cost=20.0, router_cost=0.1):
    """Simulate cost with vs without routing."""
    # Without routing: everything goes to large model
    baseline_cost = num_queries * large_cost
    
    # With routing
    num_easy = int(num_queries * easy_ratio)
    num_hard = num_queries - num_easy
    routed_cost = (
        num_queries * router_cost +  # router runs on everything
        num_easy * small_cost +       # easy → small model
        num_hard * large_cost         # hard → large model
    )
    
    return baseline_cost, routed_cost

# Sweep easy ratios
ratios = np.linspace(0.1, 0.95, 20)
savings_list = []

for ratio in ratios:
    baseline, routed = simulate_cascade(easy_ratio=ratio)
    savings_list.append((1 - routed / baseline) * 100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ratios * 100, savings_list, 'b-o', linewidth=2, markersize=4)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% savings')
ax.set_xlabel("% of queries routed to small model")
ax.set_ylabel("Cost savings (%)")
ax.set_title("Cascade routing: cost savings vs routing ratio")
ax.grid(True, alpha=0.3)
ax.legend()
ax.annotate(f'Typical: 70-80% easy\n→ {savings_list[13]:.0f}% savings',
            xy=(75, savings_list[13]), xytext=(50, savings_list[13] - 15),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.show()

baseline, routed = simulate_cascade(easy_ratio=0.75)
print(f"With 75% easy queries:")
print(f"  Baseline (all large):  {baseline:.0f} cost units")
print(f"  With routing:          {routed:.0f} cost units")
print(f"  Savings:               {(1-routed/baseline)*100:.1f}%")
print(f"  Effective cost/query:  {routed/1000:.1f} (vs {large_cost:.1f} baseline)")

## 5. Multi-Token Prediction

### The idea
Standard LLMs have one output head that predicts the next token. Multi-token prediction
models have **N independent heads** that each predict a different future position:

```
Standard:    Input → Backbone → Head → token[t+1]

Multi-token: Input → Backbone → Head 1 → token[t+1]
                              → Head 2 → token[t+2]
                              → Head 3 → token[t+3]
                              → Head 4 → token[t+4]
```

### How it accelerates inference
The extra heads act as a **built-in draft model** (self-speculative decoding):
- Head 1 generates token[t+1] authoritatively
- Heads 2–4 provide draft tokens for speculation
- Next forward pass verifies all of them in parallel
- No separate draft model needed

### Results (Meta, Gloeckle 2024)
- **Up to 3x inference speedup** (even at large batch sizes)
- 12% more HumanEval problems solved (better code generation)
- 17% improvement on MBPP (also code)
- Training requires multi-token objective from scratch (cannot retrofit)

### Tradeoff
- Requires pre-training with the multi-token objective
- Extra heads add parameters (though small relative to backbone)
- Cannot be applied to existing models without retraining

## 6. Early Exit / Adaptive Computation

### The observation
Not all tokens are equally hard. Predicting "the" after "in" requires less computation
than predicting a rare technical term. Yet standard models apply all 24 (or 128!) layers
to every token equally.

### How early exit works

```
Token: "the"  (easy)          Token: "phenomenology"  (hard)

Layer 1  → confidence: 0.3    Layer 1  → confidence: 0.1
Layer 2  → confidence: 0.5    Layer 2  → confidence: 0.1
Layer 3  → confidence: 0.8    Layer 3  → confidence: 0.2
Layer 4  → confidence: 0.95   Layer 4  → confidence: 0.3
  EXIT ✓  (skip layers 5-24)  Layer 5  → confidence: 0.4
                               ...       ...
                              Layer 24 → confidence: 0.92
                                EXIT ✓  (full computation)
```

### Implementations
- **CALM** (Microsoft): Confident Adaptive Language Modeling
- **LayerSkip** (Meta, 2024): exit + self-speculative verification

### Results
- 20–40% FLOPs reduction with <1% quality degradation
- Works best on tasks with many "easy" tokens (summarisation, translation)

### Tradeoffs
- Requires architectural modifications (exit classifiers at each layer)
- Hard to batch efficiently — different tokens exit at different layers
- Calibrating confidence thresholds is task-dependent
- Most beneficial for compute-bound scenarios (less so for memory-bound decode)

In [ ]:
# Simulate early exit savings
np.random.seed(42)

num_layers = 24
num_tokens = 200

# Simulate token difficulty: most tokens are easy, some are hard
# Easy tokens exit early, hard tokens use all layers
exit_layers = np.clip(
    np.random.exponential(scale=6, size=num_tokens).astype(int) + 1,
    1, num_layers
)

avg_layers_used = exit_layers.mean()
flops_saved = (1 - avg_layers_used / num_layers) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of exit layers
ax1.hist(exit_layers, bins=range(1, num_layers + 2), edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(x=avg_layers_used, color='red', linestyle='--', linewidth=2, label=f'Avg: {avg_layers_used:.1f} layers')
ax1.axvline(x=num_layers, color='gray', linestyle=':', linewidth=2, label=f'Full model: {num_layers} layers')
ax1.set_xlabel("Exit layer")
ax1.set_ylabel("Number of tokens")
ax1.set_title("Token exit distribution (early exit)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative compute saved
sorted_exits = np.sort(exit_layers)
cumulative_compute = np.cumsum(sorted_exits) / (np.arange(1, num_tokens + 1) * num_layers) * 100
ax2.plot(range(1, num_tokens + 1), 100 - cumulative_compute, linewidth=2, color='green')
ax2.set_xlabel("Token index (sorted by difficulty)")
ax2.set_ylabel("Cumulative FLOPs saved (%)")
ax2.set_title("FLOPs savings from early exit")
ax2.grid(True, alpha=0.3)
ax2.axhline(y=flops_saved, color='red', linestyle='--', alpha=0.5, label=f'Overall: {flops_saved:.0f}% saved')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Simulated {num_tokens} tokens through {num_layers}-layer model:")
print(f"  Average layers used: {avg_layers_used:.1f} / {num_layers}")
print(f"  FLOPs saved: {flops_saved:.0f}%")
print(f"  Tokens exiting in first 4 layers: {(exit_layers <= 4).sum()} ({(exit_layers <= 4).mean()*100:.0f}%)")
print(f"  Tokens needing all layers: {(exit_layers == num_layers).sum()} ({(exit_layers == num_layers).mean()*100:.0f}%)")

## 7. Summary — when to use what

| Strategy | Bottleneck addressed | Speedup | Quality | Complexity | Best for |
|----------|---------------------|---------|---------|-----------|----------|
| **Speculative decoding** | Memory bandwidth (decode) | 2–3.5x latency | Lossless | Medium | Low-batch, latency-sensitive |
| **Disaggregated inference** | Prefill/decode interference | 1.4–7.4x throughput | Lossless | High | Mixed workloads, large scale |
| **Prefix caching** | Redundant prefill compute | Up to 6.4x | Lossless | Low | Shared prompts, chat, RAG |
| **Cascade routing** | Cost on easy queries | 2x+ cost reduction | Near-lossless | Medium | Cost-sensitive APIs |
| **Multi-token prediction** | Sequential decoding | Up to 3x | Lossless (improved) | High (training) | New model development |
| **Early exit** | Over-compute on easy tokens | 20–40% FLOPs | <1% degradation | High | Compute-bound scenarios |

### These compose!
Production systems typically combine multiple strategies:

```
vLLM:     Continuous batching + PagedAttention + Prefix caching + Speculative decoding
SGLang:   Continuous batching + RadixAttention + Compressed FSMs + Chunked prefill
Kimi:     Disaggregated + Tiered KV cache + Prefix sharing + Overload prediction
```

The choice depends on your workload:
- **Latency-sensitive streaming**: Speculative decoding + chunked prefill
- **High-throughput batch**: Continuous batching + prefix caching + large batch decode
- **Long-context RAG**: Disaggregated + prefix caching + KV quantization
- **Cost-optimised API**: Cascade routing + prefix caching + quantization